# Teach a Taxi to pick up and drop off passengers at the right locations with Reinforcement Learning

In [2]:
%pip install gym

Defaulting to user installation because normal site-packages is not writeable
  Using cached gym-0.26.2.tar.gz (721 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached gym_notices-0.1.0-py3-none-any.whl.metadata (1.2 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached gym_notices-0.1.0-py3-none-any.whl (3.3 kB)
  Created wheel for gym: filename=gym-0.26.2-py3-none-any.whl size=827738 sha256=5cd1e7c78f8c36c90784066b157e318f31970ae1c7c8dab4142cd2dfc3429d09
  Stored in directory: c:\users\ashwi\appdata\local\pip\cache\wheels\95\51\6c\9bb05ebbe7c5cb8171dfaa3611f32622ca4658d53f31c79077
Successfully built gym
Not


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\ashwi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import gym
import numpy as np
import pickle, os

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:
env = gym.make("Taxi-v3", render_mode="ansi")

In [5]:
state = env.reset()

In [6]:
state

(72, {'prob': 1.0, 'action_mask': array([1, 0, 1, 1, 0, 0], dtype=int8)})

In [7]:
print(env.render())

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




<h1>Possible Actions</h1>

down (0), up (1), right (2), left (3), pick-up (4), and drop-off (5)

In [8]:
n_states = env.observation_space.n
n_actions = env.action_space.n

State Space BreakdownTaxi Location: The grid is 5x5, meaning there are 25 possible positions for the taxi itself (5 rows $\times$ 5 columns).Passenger Location: The passenger can be in one of 5 states. They are either waiting at one of the 4 designated locations (R, G, Y, B) or they are currently inside the taxi.Destination: The passenger wants to go to one of the 4 designated locations (R, G, Y, B).To get the total number of unique states the environment can be in, you multiply these possibilities together:

In [10]:
n_states

500

In [11]:
env.unwrapped.s = 254

In [12]:
print(env.render())

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [13]:
print(env.render())

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




<h1>How good does behaving completely random do?</h1>

In [49]:
state = env.reset()
counter = 0
g = 0
reward = None

In [50]:
while reward != 20:
    np.bool8 = np.bool_
    state, reward, done, info, _ = env.env.step(env.action_space.sample())
    counter += 1
    g += reward

In [51]:
print("Solved in {} Steps with a total reward of {}".format(counter,g))

Solved in 2815 Steps with a total reward of -11335


## Let's look at just one episode and see how the Q values change after each step using the formula below

In [18]:
Q = np.zeros([n_states, n_actions])

In [19]:
episodes = 1
alpha = 0.618

In [20]:
for episode in range(1,episodes+1):
    done = False
    reward = 0
    state,_ = env.reset()
    firstState = state
    print("Initial State = {}".format(state))
    while reward != 20:
        action = np.argmax(Q[state]) 
        state2, reward, done, truncated, info = env.env.step(action)
        Q[state,action] = Q[state,action] +  alpha * (reward + np.max(Q[state2]) - Q[state,action]) 
        state = state2

Initial State = 433


In [21]:
firstState

433

In [22]:
finalState = state
finalState

85

## Let's look at the first step:

In [23]:
firstState

433

## Let's look at the final step:

In [24]:
finalState

85

In [25]:
Q

array([[ 0.      ,  0.      ,  0.      ,  0.      ,  0.      ,  0.      ],
       [ 0.      ,  0.      ,  0.      ,  0.      ,  0.      ,  0.      ],
       [ 0.      ,  0.      ,  0.      ,  0.      ,  0.      ,  0.      ],
       ...,
       [-1.236   , -0.854076, -1.236   , -1.236   , -6.18    , -6.18    ],
       [ 0.      ,  0.      ,  0.      ,  0.      ,  0.      ,  0.      ],
       [ 0.      ,  0.      ,  0.      ,  0.      ,  0.      ,  0.      ]],
      shape=(500, 6))

## Let's run over multiple episodes so that we can converge on a optimal policy

In [26]:
episodes = 500
rewardTracker = []

In [27]:
G = 0
alpha = 0.618

In [ ]:
for episode in range(1,episodes+1):
    done = False
    G, reward = 0,0
    state,_ = env.reset()
    while done != True:
        action = np.argmax(Q[state]) 
        state2, reward, done, truncated,info = env.step(action) 

        # This is the bellman equation for Q learning
        Q[state,action] += alpha * ((reward + (np.max(Q[state2]))  - Q[state,action]))
        G += reward
        state = state2
        
    if episode % 100 == 0:
        print('Episode {} Total Reward: {}'.format(episode,G))

Episode 100 Total Reward: -20
Episode 200 Total Reward: 10
Episode 300 Total Reward: 8
Episode 400 Total Reward: -10
Episode 500 Total Reward: 3


In [29]:
Q

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-5.6823652 , -5.562     , -5.77083318, -5.562     , 11.        ,
        -6.18      ],
       [-4.03923602, -4.326     , -4.29466436, -4.326     , 15.        ,
        -6.18      ],
       ...,
       [-3.09      , -2.77696745, -3.09      , -2.82764334, -6.18      ,
        -6.18      ],
       [-4.944     , -4.93872338, -4.944     , -5.01615511, -6.18      ,
        -6.18      ],
       [-1.236     , -1.236     , -1.236     ,  6.784404  , -6.18      ,
        -6.18      ]], shape=(500, 6))

## Now that we have learned the optimal Q Values we have developed a optimal policy and have no need to train the agent anymore

In [30]:
env.render()

'+---------+\n|\x1b\x1b\x1bR\x1b\x1b\x1b: | : :G|\n| : | : : |\n| : : : : |\n| | : | : |\n|Y| : |B: |\n+---------+\n  (Dropoff)\n'

In [31]:
counter = 0
state ,_ = env.reset()
done = False

In [32]:
while done != True:
    # We simply take the action with the highest Q Value
    action = np.argmax(Q[state])
    state, reward, done, truncated, info = env.step(action)
    counter += 1
    #env.render()

In [33]:
counter

18

In [34]:
with open("smartTaxi_qTable.pkl", 'wb') as f:
    pickle.dump(Q, f)

In [35]:
with open("smartTaxi_qTable.pkl", 'rb') as f:
    Qtest = pickle.load(f)